# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 1. 作业中的 LIF 公式

作业给出的 Euler 离散更新是：

$$V_{t+\Delta t}=V_t+\frac{\Delta t}{\tau_m}\left[-(V_t-V_{\mathrm{rest}})+RI_t\right]$$

$V_t$ 是膜电位，$V_{\mathrm{rest}}$ 是静息电位，$\tau_m$ 是膜时间常数，$R$ 是电阻，$I_t$ 是输入电流。括号中的两项分别表示泄漏和输入。

In [ ]:
# 单位：电位 mV、时间 ms、电阻 MΩ、电流 nA
rest, threshold, voltage = -65.0, -50.0, -60.0
dt, tau, resistance, current = 0.1, 20.0, 10.0, 2.0

# 作业中的 Euler 更新（暂不考虑发放后的重置）
next_voltage = voltage + dt / tau * (-(voltage - rest) + resistance * current)
print(f"一步后：{next_voltage:.3f} mV")

## 2. snnTorch 的更新公式

`snn.Leaky` 在不考虑发放后重置时，使用：

$$U_{t+1}=\beta U_t+X_t$$

其中 $U_t$ 是传给神经元的膜电位，$\beta$ 控制保留多少旧电位，$X_t$ 是本步输入增量。接下来把作业公式转换成这个形式。

In [ ]:
# 简单演示：当前相对电位为 1，保留 90%，再加入 0.2
example = snn.Leaky(beta=0.9, threshold=1000, reset_delay=False)
spike, relative_voltage = example(torch.tensor(0.2), torch.tensor(1.0))
print(f"一步后相对电位：{relative_voltage.item():.1f}；脉冲：{int(spike.item())}")

## 3. 把静息电位设为零点

定义相对静息电位 $U_t=V_t-V_{\mathrm{rest}}$。例如作业里的静息电位是 −65 mV，发放阈值是 −50 mV，那么在新坐标下：

$$U_{\mathrm{rest}}=0,\qquad U_{\mathrm{th}}=-50-(-65)=15\ \mathrm{mV}$$

更新后也减去同一个静息电位：$U_{t+\Delta t}=V_{t+\Delta t}-V_{\mathrm{rest}}$。

In [ ]:
relative_voltage = voltage - rest
relative_threshold = threshold - rest
print(f"当前相对电位：{relative_voltage:.1f} mV；相对阈值：{relative_threshold:.1f} mV")

## 4. 将作业公式改写成相对电位

把 $V_t=U_t+V_{\mathrm{rest}}$ 代入作业公式，减去两侧的静息电位：

$$U_{t+\Delta t}=U_t+\frac{\Delta t}{\tau_m}(-U_t+RI_t)$$

合并 $U_t$ 的系数：

$$U_{t+\Delta t}=\left(1-\frac{\Delta t}{\tau_m}\right)U_t+\frac{\Delta t}{\tau_m}RI_t$$

与 $U_{t+1}=\beta U_t+X_t$ 逐项比较，得到：

$$\boxed{\beta=1-\frac{\Delta t}{\tau_m}},\qquad \boxed{X_t=\frac{\Delta t}{\tau_m}RI_t}$$

所以相同参数下，两者的 Euler 膜电位更新一致。

In [ ]:
beta = 1 - dt / tau
drive = dt / tau * resistance * current
relative_next = beta * relative_voltage + drive
print(f"两种写法一致：{np.isclose(relative_next + rest, next_voltage)}")

## 5. 代入默认参数

作业建议 $\Delta t=0.1$ ms、$\tau_m=20$ ms、$R=10$ MΩ。若电流为 2 nA，则：

$$\beta=1-0.1/20=0.995$$
$$X=(0.1/20)\times10\times2=0.1\ \mathrm{mV}$$

这里用到 $1\ \mathrm{M\Omega}\times1\ \mathrm{nA}=1\ \mathrm{mV}$。注意传给 `snn.Leaky` 的是 **0.1 mV 的每步增量**，不是直接传入 2 nA。

In [ ]:
print(f"beta = {beta:.3f}；每步输入增量 = {drive:.3f} mV")

## 6. 发放、重置与不应期

上面的公式只描述未发放时的更新。每步更新后还要执行：

1. 若 $V\geq V_{\mathrm{th}}$，记录一个 Spike。
2. 将电位重置到 $V_{\mathrm{reset}}=-65$ mV，对应相对电位 0。
3. 随后 2 ms 不应期保持重置电位，不再次发放；$\Delta t=0.1$ ms 时是 **20 步**。

代码通过 `reset_delay=False` 立即重置，用 `at_threshold` 将默认的严格大于改成达到阈值即发放；不应期由外部计数器实现。

In [ ]:
def at_threshold(voltage):
    # 输入为相对阈值的电位：达到阈值（包括恰好等于）时发放。
    return (voltage >= 0).to(voltage.dtype)

In [ ]:
neuron = snn.Leaky(
    beta=torch.tensor(beta, dtype=torch.float64),
    threshold=relative_threshold,
    reset_mechanism="zero",
    reset_delay=False,
    spike_grad=at_threshold,
).double()
spike, reset_voltage = neuron(
    torch.tensor(15.0, dtype=torch.float64), torch.tensor(0.0, dtype=torch.float64)
)
refractory_steps = round(2.0 / dt)
print(
    f"达到阈值：发放={bool(spike.item())}，重置后相对电位={reset_voltage.item():.1f} mV，不应期={refractory_steps} 步"
)

## 7. 实现一个 LIF 神经元

下面的 `simulate` 按上面的参数映射创建 `snn.Leaky`。输入是恒定电流，输出是每一步的膜电位和是否发放。`at_threshold` 只用于前向仿真，不用于训练。

In [ ]:
def simulate(current_na, duration_ms=2000, dt=0.1):
    rest, reset, threshold = -65.0, -65.0, -50.0  # mV
    tau, resistance, refractory_ms = 20.0, 10.0, 2.0  # ms、MΩ、ms
    neuron = snn.Leaky(
        beta=torch.tensor(1 - dt / tau, dtype=torch.float64),
        threshold=threshold - rest,
        reset_mechanism="zero",
        reset_delay=False,
        spike_grad=at_threshold,
    ).double()
    voltage = torch.tensor(reset - rest, dtype=torch.float64)
    drive = torch.tensor(dt / tau * resistance * current_na, dtype=torch.float64)
    remaining = 0
    voltages, spikes = [], []
    with torch.no_grad():
        for _ in range(round(duration_ms / dt)):
            if remaining > 0:  # 不应期：保持重置电位，不发放
                voltage = torch.tensor(reset - rest, dtype=torch.float64)
                spike = False
                remaining -= 1
            else:
                spike, voltage = neuron(drive, voltage)
                spike = bool(spike.item())
                if spike:
                    voltage = torch.tensor(reset - rest, dtype=torch.float64)
                    remaining = round(refractory_ms / dt)
            voltages.append(voltage.item() + rest)
            spikes.append(spike)
    return np.asarray(voltages), np.asarray(spikes)

## 8. 观察一次仿真

给神经元施加 2 nA 恒定电流，运行 2 秒。图中只展示前 200 ms，红虚线是发放阈值，圆点是脉冲发生的时刻。这里记录的是**每一步结束后的电位**，因此脉冲对应的电位已经重置。

In [ ]:
dt = 0.1
voltage, spikes = simulate(current_na=2.0, dt=dt)
time_ms = (np.arange(len(voltage)) + 1) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_ms, voltage, label="Membrane potential")
ax.axhline(-50, color="red", linestyle="--", label="Threshold")
ax.scatter(time_ms[spikes], voltage[spikes], s=8, label="Spike / reset")
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.set_xlim(0, 200)
ax.legend()
plt.show()
print(
    f"脉冲数：{spikes.sum()}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 9. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

这里完成的是 Task 1。不同输入、系统对照实验和滑动发放率属于后续任务。

In [ ]:
critical_current = (threshold - rest) / resistance
print(f"理论临界电流：{critical_current:.1f} nA")
for current_na in (1.0, 1.5, 2.0):
    _, spikes = simulate(current_na)
    print(f"{current_na:.1f} nA：{spikes.sum()} 个脉冲")